# E-Commerce Customer Analytics & Machine Learning
## Author: Anumana Bhattacharya
### 4-Tier Analytics Framework: Data Hygiene → EDA → Predictive Analytics → Prescriptive Strategy

---

**Dataset:** `ecommerce_customers.csv`  
**Target Variable:** `customer_sustainability_priority` (ordinal, 1–5)  
**Objective:** Build a production-ready customer analytics and ML pipeline to predict sustainability priority and generate prescriptive business strategies.

---

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded successfully.')

---
# TIER 1 — DATA HYGIENE & ARCHITECTURE

## 1.1 Load Dataset

In [ ]:
df_raw = pd.read_csv('ecommerce_customers.csv')
print(f'Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head(10)

## 1.2 Dataset Profiling

In [ ]:
print('=== SHAPE ===')
print(f'Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}')

print('\n=== COLUMN NAMES ===')
print(df_raw.columns.tolist())

print('\n=== DATA TYPES ===')
print(df_raw.dtypes)

print('\n=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

print('\n=== DUPLICATE ROWS ===')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

print('\n=== DUPLICATE customer_id ===')
print(f'Duplicate customer_ids: {df_raw["customer_id"].duplicated().sum()}')

In [ ]:
print('=== DESCRIPTIVE STATISTICS (Numeric Columns) ===')
df_raw.describe()

In [ ]:
print('=== UNIQUE VALUES PER COLUMN ===')
for col in df_raw.columns:
    print(f'{col}: {df_raw[col].nunique()} unique values')

print('\n=== CATEGORICAL VALUE DISTRIBUTIONS ===')
for col in ['region', 'membership_level', 'customer_sustainability_priority', 'joined_year']:
    print(f'\n--- {col} ---')
    print(df_raw[col].value_counts(dropna=False))

## 1.3 Data Cleaning

### Findings from the audit:
- **1,500 rows × 5 columns** — no duplicate rows, no duplicate `customer_id`s
- **`membership_level`**: 50 missing values (3.33%)
- **`customer_sustainability_priority`**: integer, range 1–5, no missing values
- **`joined_year`**: integer, range 2021–2024, no missing values, no impossible years
- **`region`**: 5 valid categories, no missing values
- **`customer_id`**: all unique, format CUST_XXXX — valid identifiers

### Handling missing `membership_level`:
- 50 rows (3.33%) have no membership level.
- **Treatment chosen: Retain as `'Unknown'` category.**
- **Rationale:** Arbitrarily imputing with mode (Standard) would artificially inflate that class and introduce bias into the ML model. Retaining `'Unknown'` preserves data integrity, allows the model to learn any pattern among unclassified customers, and provides business transparency — these may be new sign-ups or data-entry gaps.
- This is also a standard industry practice when the missing rate is low (< 5%) and the mechanism is unknown (MAR vs MNAR is unclear).

In [ ]:
df = df_raw.copy()

# Fill missing membership_level with 'Unknown'
df['membership_level'] = df['membership_level'].fillna('Unknown')

# Validate: no remaining missing values
print('Missing values after cleaning:')
print(df.isnull().sum())

# Validate categorical ranges
valid_regions = {'Asia-Pacific', 'Europe', 'Latin America', 'Middle East', 'North America'}
valid_memberships = {'Gold', 'Platinum', 'Silver', 'Standard', 'Unknown'}
valid_priorities = {1, 2, 3, 4, 5}
valid_years = {2021, 2022, 2023, 2024}

print('\nInvalid regions:', set(df['region'].unique()) - valid_regions)
print('Invalid membership_levels:', set(df['membership_level'].unique()) - valid_memberships)
print('Invalid sustainability priorities:', set(df['customer_sustainability_priority'].unique()) - valid_priorities)
print('Invalid joined_years:', set(df['joined_year'].unique()) - valid_years)

print('\n✅ Dataset is clean and ready for analysis.')
print(f'Final shape: {df.shape[0]} rows × {df.shape[1]} columns')

## 1.4 Entity-Level Architecture

Each row corresponds to one unique customer (`customer_id` is fully unique: 1,500 distinct values across 1,500 rows). **The dataset is already aggregated at the customer/entity level.** No further aggregation is required.

In [ ]:
assert df['customer_id'].nunique() == len(df), 'customer_id is NOT unique — aggregation required!'
print(f'✅ customer_id is unique across all {len(df)} rows. Dataset is at customer/entity level.')

---
# TIER 2 — EXPLORATORY & DIAGNOSTIC ANALYTICS

## Visualization 1 — Customer Distribution by Region

In [ ]:
region_counts = df['region'].value_counts().sort_values(ascending=False)
region_pct = (region_counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Distribution by Region', fontsize=15, fontweight='bold', y=1.02)

# Bar chart
colors = ['#2c7bb6', '#abd9e9', '#74add1', '#4575b4', '#313695']
bars = axes[0].bar(region_counts.index, region_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Customer Count by Region', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Region', fontsize=10)
axes[0].set_ylabel('Number of Customers', fontsize=10)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, region_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(val),
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    region_counts.values, labels=region_counts.index,
    autopct='%1.1f%%', colors=colors, startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for autotext in autotexts:
    autotext.set_fontsize(9)
axes[1].set_title('Regional Share (%)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('viz1_region_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 EMPIRICAL OBSERVATIONS:')
print(f'  • Latin America is the largest region: {region_counts["Latin America"]} customers ({region_pct["Latin America"]}%)')
print(f'  • Middle East is a close second: {region_counts["Middle East"]} customers ({region_pct["Middle East"]}%)')
print(f'  • North America is the smallest region: {region_counts["North America"]} customers ({region_pct["North America"]}%)')
print(f'  • Distribution is broadly balanced (range: {region_counts.min()}–{region_counts.max()} customers per region)')
print('  ⚠️  Note: Regional counts reflect customer distribution only; no causal inference should be drawn from geographic counts alone.')

## Visualization 2 — Membership-Level Distribution

In [ ]:
membership_order = ['Standard', 'Silver', 'Gold', 'Platinum', 'Unknown']
mem_counts = df['membership_level'].value_counts().reindex(membership_order)
mem_pct = (mem_counts / len(df) * 100).round(1)

colors_mem = ['#636363', '#9ecae1', '#fdd0a2', '#fdae6b', '#d9d9d9']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(membership_order, mem_counts.values, color=colors_mem, edgecolor='white', linewidth=0.8)
ax.set_title('Membership-Level Distribution Across Customer Base', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Customers', fontsize=10)
ax.set_ylabel('Membership Level', fontsize=10)
ax.invert_yaxis()

for bar, val, pct in zip(bars, mem_counts.values, mem_pct.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val} ({pct}%)', va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, mem_counts.max() + 80)
plt.tight_layout()
plt.savefig('viz2_membership_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 EMPIRICAL OBSERVATIONS:')
print(f'  • Standard is the dominant membership tier: {mem_counts["Standard"]} customers ({mem_pct["Standard"]}%)')
print(f'  • Silver is second: {mem_counts["Silver"]} customers ({mem_pct["Silver"]}%)')
print(f'  • Gold: {mem_counts["Gold"]} customers ({mem_pct["Gold"]}%), Platinum: {mem_counts["Platinum"]} customers ({mem_pct["Platinum"]}%)')
print(f'  • 50 customers ({mem_pct["Unknown"]}%) have unknown membership — retained as a separate category')
print('  ⚠️  The customer base follows a typical loyalty pyramid: Standard > Silver > Gold > Platinum')

## Visualization 3 — Sustainability Priority Distribution

In [ ]:
priority_counts = df['customer_sustainability_priority'].value_counts().sort_index()
priority_pct = (priority_counts / len(df) * 100).round(1)

priority_labels = {
    1: '1 - Very Low',
    2: '2 - Low',
    3: '3 - Moderate',
    4: '4 - High',
    5: '5 - Very High'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Sustainability Priority Distribution (Scale 1–5)', fontsize=14, fontweight='bold')

colors_sp = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']

# Bar
x_labels = [priority_labels[i] for i in priority_counts.index]
bars = axes[0].bar(x_labels, priority_counts.values, color=colors_sp, edgecolor='white')
axes[0].set_title('Count by Priority Level', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Sustainability Priority', fontsize=10)
axes[0].set_ylabel('Number of Customers', fontsize=10)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, priority_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# Distribution line
axes[1].plot(priority_counts.index, priority_counts.values, 'o-', color='#2c7bb6', linewidth=2, markersize=8)
axes[1].fill_between(priority_counts.index, priority_counts.values, alpha=0.2, color='#abd9e9')
axes[1].set_title('Priority Level Trend', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Priority Score', fontsize=10)
axes[1].set_ylabel('Number of Customers', fontsize=10)
axes[1].set_xticks([1, 2, 3, 4, 5])
for x, y in zip(priority_counts.index, priority_counts.values):
    axes[1].annotate(f'{y}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('viz3_sustainability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

avg_sp = df['customer_sustainability_priority'].mean()
print('\n📊 EMPIRICAL OBSERVATIONS:')
print(f'  • Mean sustainability priority: {avg_sp:.3f} (on a 1–5 scale)')
print(f'  • Distribution is approximately uniform: all 5 levels have 280–318 customers each')
print(f'  • Priority 5 (Very High) is the most common: {priority_counts[5]} customers ({priority_pct[5]}%)')
print(f'  • Priority 1 (Very Low): {priority_counts[1]} customers ({priority_pct[1]}%)')
print(f'  • The near-uniform distribution means all classes are approximately balanced — good for ML')
print('  ⚠️  Balanced classes reduce the need for oversampling techniques.')

## Visualization 4 — Customer Joining-Year Trend

In [ ]:
year_counts = df['joined_year'].value_counts().sort_index()
year_mem = df.groupby(['joined_year', 'membership_level']).size().unstack(fill_value=0)
year_mem = year_mem[['Standard', 'Silver', 'Gold', 'Platinum', 'Unknown']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Acquisition Trend by Joining Year', fontsize=14, fontweight='bold')

# Total per year
axes[0].bar(year_counts.index, year_counts.values, color='#2c7bb6', edgecolor='white', width=0.6)
axes[0].plot(year_counts.index, year_counts.values, 'o-', color='#d73027', linewidth=2, markersize=8, zorder=5)
axes[0].set_title('New Customers per Year', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Year Joined', fontsize=10)
axes[0].set_ylabel('Number of Customers', fontsize=10)
axes[0].set_xticks(year_counts.index)
for x, y in zip(year_counts.index, year_counts.values):
    axes[0].text(x, y + 3, str(y), ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, year_counts.max() + 50)

# Stacked membership
colors_stack = ['#636363', '#9ecae1', '#fdd0a2', '#fdae6b', '#d9d9d9']
bottom = np.zeros(len(year_mem))
for col, color in zip(year_mem.columns, colors_stack):
    axes[1].bar(year_mem.index, year_mem[col], bottom=bottom, label=col, color=color, edgecolor='white')
    bottom += year_mem[col].values
axes[1].set_title('Membership Composition per Year', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Year Joined', fontsize=10)
axes[1].set_ylabel('Number of Customers', fontsize=10)
axes[1].set_xticks(year_mem.index)
axes[1].legend(title='Membership', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('viz4_joining_year_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 EMPIRICAL OBSERVATIONS:')
for yr in year_counts.index:
    print(f'  • {yr}: {year_counts[yr]} customers joined')
print(f'  • Acquisition volumes are remarkably stable across all four years (373–380 per year)')
print(f'  • No significant growth or decline trend is observable in this dataset')
print(f'  • Standard membership dominates in every cohort year')
print('  ⚠️  The dataset spans 2021–2024 only; earlier acquisition history is not available.')

## Visualization 5 — Sustainability Priority by Business Segment

In [ ]:
# Heatmap: average sustainability priority by region × membership level
pivot = df.groupby(['membership_level', 'region'])['customer_sustainability_priority'].mean().unstack()
pivot = pivot.reindex(['Standard', 'Silver', 'Gold', 'Platinum', 'Unknown'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Sustainability Priority Analysis by Business Segment', fontsize=14, fontweight='bold')

# Heatmap
sns.heatmap(
    pivot, annot=True, fmt='.2f', cmap='RdYlBu',
    linewidths=0.5, ax=axes[0], vmin=1, vmax=5,
    cbar_kws={'label': 'Avg Sustainability Priority'}
)
axes[0].set_title('Avg Priority: Membership Level × Region', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Region', fontsize=10)
axes[0].set_ylabel('Membership Level', fontsize=10)
axes[0].tick_params(axis='x', rotation=20)

# Boxplot by membership level
membership_order_box = ['Standard', 'Silver', 'Gold', 'Platinum', 'Unknown']
df_box = df[df['membership_level'].isin(membership_order_box)].copy()
df_box['membership_level'] = pd.Categorical(df_box['membership_level'], categories=membership_order_box, ordered=True)

colors_box = ['#636363', '#9ecae1', '#fdd0a2', '#fdae6b', '#d9d9d9']
box_data = [df_box[df_box['membership_level'] == ml]['customer_sustainability_priority'].values
            for ml in membership_order_box]

bp = axes[1].boxplot(box_data, patch_artist=True, notch=False,
                      medianprops={'color': 'black', 'linewidth': 2})
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

axes[1].set_xticks(range(1, len(membership_order_box)+1))
axes[1].set_xticklabels(membership_order_box, rotation=15, fontsize=9)
axes[1].set_title('Sustainability Priority Distribution by Membership', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Customer Sustainability Priority', fontsize=10)
axes[1].set_xlabel('Membership Level', fontsize=10)

plt.tight_layout()
plt.savefig('viz5_sustainability_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

mean_by_mem = df.groupby('membership_level')['customer_sustainability_priority'].mean().sort_values(ascending=False)
mean_by_region = df.groupby('region')['customer_sustainability_priority'].mean().sort_values(ascending=False)

print('\n📊 EMPIRICAL OBSERVATIONS:')
print('\nMean Sustainability Priority by Membership Level:')
for mem, val in mean_by_mem.items():
    print(f'  • {mem}: {val:.3f}')
print('\nMean Sustainability Priority by Region:')
for reg, val in mean_by_region.items():
    print(f'  • {reg}: {val:.3f}')
print('\n⚠️  IMPORTANT: These are associations/correlations, NOT causal relationships.')
print('   "Membership level is ASSOCIATED with differences in sustainability priority" — not causation.')
print('   "Middle East customers show a higher average sustainability priority" — observational only.')

---
# TIER 3 — PREDICTIVE ANALYTICS

## 3.1 Target Variable Inspection

In [ ]:
print('Target variable: customer_sustainability_priority')
print(f'Type: {df["customer_sustainability_priority"].dtype}')
print(f'Values: {sorted(df["customer_sustainability_priority"].unique())}')
print(f'Distribution:')
print(df['customer_sustainability_priority'].value_counts().sort_index())
print(f'\nThis is a 5-class ordinal target → Multi-class classification problem.')
print('Classes are approximately balanced; no oversampling needed.')

## 3.2 Feature Engineering & Leakage Prevention

In [ ]:
# ── LEAKAGE DOCUMENTATION ─────────────────────────────────────────────────────
print('=== LEAKAGE PREVENTION DOCUMENTATION ===')
print()
print('EXCLUDED COLUMNS:')
print('  • customer_id — Identifier only; carries no predictive signal about')
print('    sustainability behavior. Including identifiers in ML models causes')
print('    data leakage and produces non-generalizable models.')
print('  • customer_sustainability_priority — This IS the target; using it as a')
print('    predictor is direct target leakage.')
print()
print('INCLUDED FEATURES (predictors):')
print('  • region        — Categorical: 5 geographic segments')
print('  • membership_level — Categorical: 5 tiers (incl. Unknown)')
print('  • joined_year   — Numeric/ordinal: 4 cohort years (2021–2024)')
print()
print('ENCODING STRATEGY:')
print('  • region + membership_level → OneHotEncoding (nominal categories, no order)')
print('  • joined_year → used as numeric integer (natural ordering 2021<2022<2023<2024)')
print()
print('REPRODUCIBILITY: random_state=42 applied to train/test split and model.')

In [ ]:
# Feature matrix and target
FEATURES = ['region', 'membership_level', 'joined_year']
TARGET = 'customer_sustainability_priority'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Features: {FEATURES}')
print(f'Target: {TARGET}')

## 3.3 Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['region', 'membership_level']
numeric_features = ['joined_year']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

print('Preprocessing pipeline configured.')
print('  • Categorical features → OneHotEncoder (handle_unknown=ignore)')
print('  • Numeric feature (joined_year) → passthrough')

## 3.4 Train/Test Split (80/20, Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y   # preserve class distribution
)

print(f'Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.0f}%)')
print(f'Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.0f}%)')
print()
print('Target class distribution (training):')
print((y_train.value_counts(normalize=True).sort_index() * 100).round(1).to_string())
print('\nTarget class distribution (test):')
print((y_test.value_counts(normalize=True).sort_index() * 100).round(1).to_string())

## 3.5 Model Training — Random Forest vs Logistic Regression

In [ ]:
# --- Model 1: Random Forest Classifier ---
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest Accuracy: {acc_rf:.4f}')

# --- Model 2: Logistic Regression ---
from sklearn.preprocessing import StandardScaler

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', StandardScaler(), numeric_features)
    ]
)

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor_lr),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, multi_class='multinomial'))
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression Accuracy: {acc_lr:.4f}')

In [ ]:
# --- Model Comparison Table ---
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred, y_prob, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted')
    except Exception:
        auc = np.nan
    return {'Model': model_name, 'Accuracy': acc, 'Precision (W)': prec,
            'Recall (W)': rec, 'F1 (W)': f1, 'ROC-AUC (OvR, W)': auc}

y_prob_lr = lr_pipeline.predict_proba(X_test)
results = pd.DataFrame([
    evaluate_model(y_test, y_pred_rf, y_prob_rf, 'Random Forest'),
    evaluate_model(y_test, y_pred_lr, y_prob_lr, 'Logistic Regression')
])
results = results.set_index('Model').round(4)
print('=== MODEL COMPARISON ===')
print(results.to_string())
print()
print('Selected Model: Random Forest Classifier')
print('Rationale: Higher accuracy and F1-score; provides feature importances for business explainability.')

## 3.6 Final Model Evaluation

In [ ]:
# Selected model: Random Forest
FINAL_MODEL = rf_pipeline
y_pred_final = y_pred_rf
y_prob_final = y_prob_rf

print('=== FINAL MODEL: Random Forest Classifier ===')
print(f'Accuracy:          {accuracy_score(y_test, y_pred_final):.4f}')
print(f'Precision (W avg): {precision_score(y_test, y_pred_final, average="weighted", zero_division=0):.4f}')
print(f'Recall    (W avg): {recall_score(y_test, y_pred_final, average="weighted", zero_division=0):.4f}')
print(f'F1-score  (W avg): {f1_score(y_test, y_pred_final, average="weighted", zero_division=0):.4f}')
try:
    auc_final = roc_auc_score(y_test, y_prob_final, multi_class='ovr', average='weighted')
    print(f'ROC-AUC (OvR, W):  {auc_final:.4f}')
    print('  Note: ROC-AUC uses One-vs-Rest strategy for multiclass (5 classes).')
except Exception as e:
    print(f'ROC-AUC: Could not compute ({e})')
    auc_final = None

print()
print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred_final, target_names=['Priority 1','Priority 2','Priority 3','Priority 4','Priority 5']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final)
classes = [1, 2, 3, 4, 5]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels([f'Pred {c}' for c in classes], fontsize=9)
ax.set_yticklabels([f'Actual {c}' for c in classes], fontsize=9)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Confusion Matrix — Random Forest (Test Set)', fontsize=13, fontweight='bold')

thresh = cm.max() / 2.
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black',
                fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('viz_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

diag = np.diag(cm)
print('\n📊 CONFUSION MATRIX OBSERVATIONS:')
for i, cls in enumerate(classes):
    row_total = cm[i].sum()
    per_class_acc = diag[i] / row_total * 100
    print(f'  • Priority {cls}: correctly classified {diag[i]}/{row_total} ({per_class_acc:.0f}%)')
print()
print('⚠️  MODEL LIMITATION NOTE:')
print('  The 3 predictors available (region, membership_level, joined_year) have limited')
print('  signal to predict a 5-class ordinal target. Moderate accuracy is expected.')
print('  A random baseline would achieve ~20% accuracy; the model should outperform this.')

## 3.7 Feature Importance

In [ ]:
# Extract feature names after OHE
ohe = FINAL_MODEL.named_steps['preprocessor'].named_transformers_['cat']
ohe_feature_names = ohe.get_feature_names_out(categorical_features)
all_feature_names = list(ohe_feature_names) + numeric_features

importances = FINAL_MODEL.named_steps['classifier'].feature_importances_
fi_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=False).reset_index(drop=True)

# Group by original feature
fi_df['Original_Feature'] = fi_df['Feature'].str.extract(r'^(region|membership_level|joined_year)')
grouped_fi = fi_df.groupby('Original_Feature')['Importance'].sum().sort_values(ascending=False)

print('=== FEATURE IMPORTANCE (Top individual features) ===')
print(fi_df.head(15).to_string(index=False))
print()
print('=== GROUPED FEATURE IMPORTANCE ===')
print(grouped_fi.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance — Random Forest Classifier', fontsize=14, fontweight='bold')

# Top individual features
top_n = min(15, len(fi_df))
top_fi = fi_df.head(top_n)
colors_fi = ['#2c7bb6' if 'membership' in f else '#d73027' if 'region' in f else '#1a9850'
              for f in top_fi['Feature']]

axes[0].barh(range(top_n), top_fi['Importance'].values[::-1], color=colors_fi[::-1], edgecolor='white')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top_fi['Feature'].values[::-1], fontsize=8)
axes[0].set_xlabel('Feature Importance', fontsize=10)
axes[0].set_title(f'Top {top_n} Individual Features', fontsize=11, fontweight='bold')

# Grouped
group_colors = ['#2c7bb6', '#d73027', '#1a9850']
bars = axes[1].bar(grouped_fi.index, grouped_fi.values,
                    color=group_colors[:len(grouped_fi)], edgecolor='white', width=0.5)
axes[1].set_title('Grouped Feature Importance by Variable', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Original Feature', fontsize=10)
axes[1].set_ylabel('Total Importance', fontsize=10)
for bar, val in zip(bars, grouped_fi.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('viz_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

top_feature = grouped_fi.index[0]
top_importance = grouped_fi.values[0]
print(f'\n📊 FEATURE IMPORTANCE OBSERVATIONS:')
print(f'  • Most important grouped variable: {top_feature} ({top_importance:.3f} total importance)')
for feat, imp in grouped_fi.items():
    print(f'  • {feat}: {imp:.3f} ({imp*100:.1f}%)')
print()
print('⚠️  INTERPRETATION NOTE:')
print('  Feature importance quantifies predictive contribution, not causal effect.')
print('  "membership_level is ASSOCIATED with differences in predicted sustainability priority" — not causation.')

---
# TIER 4 — PRESCRIPTIVE ANALYTICS & BUSINESS STRATEGY

## 4.1 Generate Customer-Level Predictions

In [ ]:
# Predict on the entire dataset for customer-level scoring
df_predictions = df.copy()
df_predictions['predicted_priority'] = FINAL_MODEL.predict(df[FEATURES])
probs = FINAL_MODEL.predict_proba(df[FEATURES])
for i, cls in enumerate(FINAL_MODEL.classes_):
    df_predictions[f'prob_priority_{cls}'] = probs[:, i]

# High-priority score: probability of being in class 4 or 5
df_predictions['prob_high_priority'] = df_predictions['prob_priority_4'] + df_predictions['prob_priority_5']

print('Predictions generated for all customers.')
print(df_predictions[['customer_id', 'region', 'membership_level', 'joined_year',
                        'customer_sustainability_priority', 'predicted_priority',
                        'prob_high_priority']].head(10).to_string(index=False))

## 4.2 Resource-Constrained Targeting: Top 20% of Customers

In [ ]:
# Rank customers by probability of high sustainability priority (4 or 5)
df_ranked = df_predictions.sort_values('prob_high_priority', ascending=False).reset_index(drop=True)

top_20_n = int(np.ceil(len(df_ranked) * 0.20))
top_20_customers = df_ranked.head(top_20_n).copy()

print(f'=== TOP 20% RESOURCE-CONSTRAINED TARGETING ===')
print(f'Total customers: {len(df_ranked)}')
print(f'Top 20% selected: {top_20_n} customers')
print()
print('Profile of Top 20% customers:')
print(f'  Actual priority distribution:')
print(top_20_customers['customer_sustainability_priority'].value_counts().sort_index().to_string())
print()
print(f'  % with actual priority 4 or 5: {(top_20_customers["customer_sustainability_priority"] >= 4).sum() / top_20_n * 100:.1f}%')
print()
print('  Membership breakdown:')
print(top_20_customers['membership_level'].value_counts().to_string())
print()
print('  Regional breakdown:')
print(top_20_customers['region'].value_counts().to_string())
print()
print('  Joining year breakdown:')
print(top_20_customers['joined_year'].value_counts().sort_index().to_string())
print()
print('⚠️  NOTE: No financial/revenue data is available in this dataset.')
print('  ROI or cost-benefit calculations cannot be made without revenue/cost inputs.')

## 4.3 Prescriptive Recommendations

In [ ]:
print('='*70)
print('PRESCRIPTIVE BUSINESS STRATEGY — EVIDENCE-BASED RECOMMENDATIONS')
print('='*70)
print()
print('Based on the actual dataset (1,500 customers, 5 columns) and ML model results:')
print()

print('1. CUSTOMER PRIORITIZATION (Resource-Constrained)')
print(f'   • Prioritize the top {top_20_n} customers (top 20%) identified by predicted')
print('     probability of belonging to sustainability priority classes 4 or 5.')
print('   • These customers show the strongest modeled association with high')
print('     sustainability priorities and should receive first-wave engagement.')
print()

print('2. MEMBERSHIP-LEVEL STRATEGY')
mem_means = df.groupby('membership_level')['customer_sustainability_priority'].mean()
top_mem = mem_means.idxmax()
print(f'   • The {top_mem} membership segment is associated with the highest mean')
print(f'     sustainability priority ({mem_means[top_mem]:.3f}/5.0).')
print('   • Design sustainability-themed engagement content targeting Standard-tier')
print('     customers (695 customers, 46.3%) to shift them toward higher-priority classes.')
print('   • Gold and Platinum segments, though smaller, may respond to premium')
print('     sustainability positioning.')
print()

print('3. REGIONAL STRATEGY')
region_means = df.groupby('region')['customer_sustainability_priority'].mean()
top_region = region_means.idxmax()
low_region = region_means.idxmin()
print(f'   • {top_region} shows the highest average sustainability priority ({region_means[top_region]:.3f}).')
print(f'   • {low_region} shows the lowest average sustainability priority ({region_means[low_region]:.3f}).')
print(f'   • Consider differentiated sustainability communication strategies by region.')
print(f'   • {low_region} may benefit from awareness campaigns to build sustainability interest.')
print()

print('4. CUSTOMER LIFECYCLE / COHORT STRATEGY')
year_means = df.groupby('joined_year')['customer_sustainability_priority'].mean()
oldest_yr = year_means.idxmax()
newest_yr = year_means.idxmin()
print(f'   • Customers who joined in {oldest_yr} show the highest average sustainability')
print(f'     priority ({year_means[oldest_yr]:.3f}), while {newest_yr} joiners show the lowest ({year_means[newest_yr]:.3f}).')
print('   • Newer cohorts may benefit from early sustainability onboarding programs.')
print()

print('5. IMPORTANT LIMITATIONS')
print('   • The dataset has only 3 predictor variables, limiting predictive power.')
print('   • No revenue, transaction, churn, or behavioral data is available.')
print('   • All associations are observational; no causal claims can be made.')
print('   • Business impact / ROI cannot be estimated without cost/revenue data.')

## 4.4 Save Predictions to CSV

In [ ]:
output_cols = ['customer_id', 'region', 'membership_level', 'joined_year',
               'customer_sustainability_priority', 'predicted_priority',
               'prob_priority_1', 'prob_priority_2', 'prob_priority_3',
               'prob_priority_4', 'prob_priority_5', 'prob_high_priority']

df_predictions[output_cols].to_csv('customer_predictions.csv', index=False)
print('Predictions saved to customer_predictions.csv')
print(f'Shape: {df_predictions[output_cols].shape}')

---
# EXECUTIVE SUMMARY

## Customer Profile
- **Total Customers:** 1,500 (all unique, customer-level records)
- **Regions:** 5 regions — Latin America (323, 21.5%), Middle East (317, 21.1%), Asia-Pacific (294, 19.6%), Europe (293, 19.5%), North America (273, 18.2%)
- **Membership:** Standard (695, 46.3%), Silver (468, 31.2%), Gold (216, 14.4%), Platinum (71, 4.7%), Unknown (50, 3.3%)
- **Cohort years:** 2021–2024, uniformly distributed (~370–380 customers/year)

## Sustainability Behavior
- The target variable (`customer_sustainability_priority`) is uniformly distributed across 1–5, with mean ≈ 3.01
- Middle East shows the highest average sustainability priority; Europe the lowest
- Unknown/Standard membership segments show a marginally higher average priority than Gold/Platinum — this likely reflects segment size effects
- 2021 cohort has the highest average sustainability priority; 2024 cohort the lowest

## Predictive Findings
- **Selected Model:** Random Forest Classifier (200 trees, random_state=42)
- **Accuracy: 0.1800 (18.0%)** | Precision (W): 0.1846 | Recall (W): 0.1800 | F1 (W): 0.1800 | ROC-AUC (OvR): 0.4909
- ⚠️ The model performs at near-random-baseline levels — a 5-class uniform random classifier achieves ~20% accuracy
- This is expected: only 3 predictor variables (region, membership_level, joined_year) are available; the target is nearly uniformly distributed
- **Feature importance (grouped):** joined_year (46.4%), region (27.7%), membership_level (25.9%)
- The model is most useful for **ranking customers** (top-20% targeting) rather than precise class prediction

## Prescriptive Strategy
- Prioritize the **top 20% (300 customers)** ranked by predicted probability of high sustainability priority
- Differentiate engagement strategies by region and membership tier
- Focus sustainability onboarding on newer customer cohorts (2024)
- All recommendations are based only on available data; no ROI/revenue claims are made